# Análisis en frecuencia

Hasta ahora las señales se han visto en función del tiempo. También se pueden ver en función de la frecuencia: cualquier señal se puede escribir como una suma de senoidales, y la transformada de Fourier indica qué frecuencias contiene y con qué amplitud. Para reconocer actividades esto es muy útil, porque muchas son rítmicas; al caminar, por ejemplo, el brazo se balancea a un ritmo casi constante.

## Parámetros

In [ ]:
CARPETA = "datos/demo"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal

from wt9011 import cargar_grabaciones
from procesamiento import agregar_magnitudes, preprocesar

grabaciones = cargar_grabaciones(CARPETA)
for g in grabaciones:
    agregar_magnitudes(g["df"])                   # señal cruda, con magnitudes
    g["pre"] = preprocesar(g["df"], g["fs"])     # señal preprocesada

ETIQUETAS = sorted(set(g["etiqueta"] for g in grabaciones))
COLORES = {e: f"C{i}" for i, e in enumerate(ETIQUETAS)}

## La idea con una señal conocida

Se construye una señal con dos senoidales, una de 2 Hz y amplitud 1 y otra de 7 Hz y amplitud 0.5, más ruido. En el tiempo cuesta distinguirlas; en frecuencia aparecen como dos picos.

In [ ]:
fs = 50
t = np.arange(0, 4, 1 / fs)
ruido = 0.3 * np.random.default_rng(0).standard_normal(len(t))
x = np.sin(2 * np.pi * 2 * t) + 0.5 * np.sin(2 * np.pi * 7 * t) + ruido

X = np.fft.rfft(x)
f = np.fft.rfftfreq(len(x), 1 / fs)
amplitud = 2 * np.abs(X) / len(x)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 5))
ax1.plot(t, x, lw=0.8)
ax1.set(xlabel="Tiempo [s]", ylabel="x")
ax2.plot(f, amplitud, lw=1)
ax2.set(xlabel="Frecuencia [Hz]", ylabel="Amplitud")
ax1.grid(alpha=0.3)
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Detalles del cálculo:

- `rfft` calcula la transformada solo para frecuencias positivas, que es suficiente cuando la señal es real.
- `rfftfreq` da el eje de frecuencias, de 0 a fs/2.
- Multiplicar por 2 y dividir entre el número de muestras hace que la altura de cada pico sea la amplitud de la senoidal.
- La separación entre frecuencias, o resolución, es Δf = fs / N = 1 / duración. Con 4 segundos, Δf = 0.25 Hz. Para distinguir dos frecuencias cercanas se necesita una señal más larga, no una fs más alta.

## FFT de una grabación

Se usa la aceleración del cuerpo en z de la primera grabación. Al quitar la gravedad, la señal oscila alrededor de cero y no aparece un pico grande en 0 Hz.

In [ ]:
g = grabaciones[0]
fs = g["fs"]
x = g["pre"]["az_cuerpo"].to_numpy()

X = np.fft.rfft(x)
f = np.fft.rfftfreq(len(x), 1 / fs)
amplitud = 2 * np.abs(X) / len(x)

f_dominante = f[np.argmax(amplitud)]
print(f"{g['archivo']}: {len(x)} muestras, {len(x) / fs:.1f} s")
print(f"Resolución: {fs / len(x):.2f} Hz")
print(f"Frecuencia dominante: {f_dominante:.2f} Hz")

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(f, amplitud, lw=1, color=COLORES[g["etiqueta"]])
ax.axvline(f_dominante, color="k", lw=0.8, ls="--")
ax.set(xlabel="Frecuencia [Hz]", ylabel="Amplitud [g]", xlim=(0, 10), title=g["archivo"])
ax.grid(alpha=0.3)
plt.show()

Con el sensor en la muñeca, el pico principal de caminar corresponde al balanceo del brazo. Un ciclo completo del brazo, adelante y atrás, dura dos pasos, así que la frecuencia de los pasos es el doble. Con un pico en 0.8 Hz serían 1.6 pasos por segundo, unos 96 por minuto.

Los picos más pequeños en múltiplos de esa frecuencia se llaman armónicos. Aparecen porque el movimiento se repite pero no tiene forma de senoidal pura.

## Densidad espectral de potencia

El espectro de una sola FFT sobre toda la grabación es ruidoso: cada frecuencia se estima una sola vez. El método de Welch divide la señal en segmentos traslapados, calcula el espectro de cada uno y los promedia. El resultado es más suave, a cambio de menos resolución, porque ahora Δf = fs / nperseg, donde nperseg es el tamaño de cada segmento.

Lo que se obtiene es la densidad espectral de potencia (PSD), en g²/Hz: cuánta potencia hay por cada hertz. El área bajo la curva en una banda es la potencia de la señal en esa banda. Se grafica en escala logarítmica porque los valores cubren varios órdenes de magnitud.

In [ ]:
NPERSEG = 256   # muestras por segmento, unos 5 s a 50 Hz

f1, P1 = signal.periodogram(x, fs=fs)
f2, P2 = signal.welch(x, fs=fs, nperseg=NPERSEG)

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.semilogy(f1, P1, lw=0.8, color="0.6", label="una sola FFT (periodograma)")
ax.semilogy(f2, P2, lw=1.5, color="C0", label=f"Welch, nperseg = {NPERSEG}")
ax.set(xlabel="Frecuencia [Hz]", ylabel="PSD [g²/Hz]", xlim=(0, fs / 2), ylim=(1e-8, None))
ax.legend(loc="upper right")
ax.grid(alpha=0.3)
plt.show()

## Comparación entre actividades

Para cada actividad se calcula la PSD de |a| en cada grabación y se promedian. Aquí se usa la señal cruda, sin el pasa-bajas, para ver todo el contenido hasta fs/2. Welch quita la media de cada segmento, así que la gravedad no aparece como un pico en 0 Hz.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for etiqueta in ETIQUETAS:
    psds = []
    for g in grabaciones:
        if g["etiqueta"] == etiqueta:
            f, P = signal.welch(g["df"]["a_mag"], fs=g["fs"], nperseg=NPERSEG)
            psds.append(P)
    ax.semilogy(f, np.mean(psds, axis=0), lw=1.5, color=COLORES[etiqueta], label=etiqueta)

ax.axvline(10, color="k", lw=0.8, ls="--")
ax.text(10.2, 1e-1, "corte del pasa-bajas")
ax.set(xlabel="Frecuencia [Hz]", ylabel="PSD de |a| [g²/Hz]", xlim=(0, 25))
ax.legend(loc="upper right")
ax.grid(alpha=0.3)
plt.show()

Las actividades con movimiento tienen casi toda su potencia por debajo de 5 Hz, y ahí están varios órdenes de magnitud por encima de "quieto". Esa diferencia de potencia en baja frecuencia es una de las características más útiles para separarlas.

En "quieto" la potencia sube por encima de 15 Hz, con picos entre 18 y 22 Hz, donde incluso supera a las otras actividades. Es la oscilación corta que se veía después de cada pulso en `03_visualizacion`: una vibración mecánica del sensor y su sujeción, no un movimiento de la persona. Está cerca de fs/2 = 25 Hz. Si el sensor vibrara por encima de 25 Hz, esa componente no desaparecería: por aliasing aparecería reflejada por debajo de 25 Hz, mezclada con la señal. Muestrear a 100 Hz daría margen para ver estas vibraciones sin aliasing, pero para las actividades que interesan no hace falta, y el pasa-bajas de 10 Hz las elimina.

## Espectrograma

La FFT dice qué frecuencias hay, pero no cuándo aparecen. El espectrograma calcula la FFT en ventanas cortas que se desplazan a lo largo de la señal y muestra el resultado como una imagen: tiempo en el eje horizontal, frecuencia en el vertical y potencia en color.

Para verlo se pega una grabación cruda de cada actividad, una detrás de otra, como si la persona hubiera cambiado de actividad sin detener la grabación.

In [ ]:
VENTANA = 64   # muestras, alrededor de 1.3 s

tramos = []
for etiqueta in ETIQUETAS:
    for g in grabaciones:
        if g["etiqueta"] == etiqueta:
            tramos.append(g["df"]["a_mag"].to_numpy())
            break
x = np.concatenate(tramos)
fs = grabaciones[0]["fs"]

f, t, S = signal.spectrogram(x, fs=fs, nperseg=VENTANA, noverlap=VENTANA - 8)
S_db = 10 * np.log10(S + 1e-12)

fig, ax = plt.subplots(figsize=(12, 4))
imagen = ax.pcolormesh(t, f, S_db, shading="auto", cmap="viridis", vmin=S_db.max() - 60)
fig.colorbar(imagen, ax=ax, label="Potencia [dB]")

inicio = 0
for etiqueta, tramo in zip(ETIQUETAS, tramos):
    ax.axvline(inicio / fs, color="w", lw=1)
    ax.text(inicio / fs + 0.8, 9.2, etiqueta, color="w")
    inicio += len(tramo)

ax.set(xlabel="Tiempo [s]", ylabel="Frecuencia [Hz]", xlim=(t[0], t[-1]), ylim=(0, 10))
plt.show()

Cada actividad deja una huella distinta: casi nada de potencia cuando la persona está quieta y bandas en baja frecuencia cuando hay movimiento, en posiciones distintas según el ritmo de cada actividad. Con el espectrograma se puede ver en qué momento cambia la actividad.

El tamaño de la ventana es un compromiso. Una ventana corta ubica bien los cambios en el tiempo pero tiene poca resolución en frecuencia, y una larga hace lo contrario. No se pueden tener las dos cosas a la vez; con `VENTANA = 32` y `VENTANA = 256` se nota la diferencia.

## Frecuencia dominante de cada grabación

La frecuencia del pico más alto de la PSD resume en un número el ritmo de la actividad. Se calcula con |ω| preprocesada, ignorando lo que está por debajo de 0.3 Hz.

In [ ]:
filas = []
for g in grabaciones:
    f, P = signal.welch(g["pre"]["g_mag"], fs=g["fs"], nperseg=NPERSEG)
    validas = f > 0.3
    f_dominante = f[validas][np.argmax(P[validas])]
    filas.append([g["archivo"], g["etiqueta"], round(f_dominante, 2)])

pd.DataFrame(filas, columns=["archivo", "etiqueta", "frecuencia dominante [Hz]"])

En las actividades rítmicas la frecuencia dominante es muy estable entre repeticiones, y por eso sirve para distinguirlas. En "quieto" no hay ritmo: el pico más alto cae en cualquier lado y el valor cambia de una grabación a otra. Esta frecuencia, junto con la potencia en distintas bandas, será parte de las características que se calculan por ventana en los siguientes notebooks.

En `07_ventaneo` las grabaciones se dividen en ventanas cortas, que serán los ejemplos para el modelo.